<a href="https://colab.research.google.com/github/shikhak28/Garbage-Waste-Object-Detection-YOLOv8/blob/main/garbage_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ultralytics huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.5 MB/s eta 0:00:00


In [2]:
import json, os, shutil, glob, zipfile

from huggingface_hub import hf_hub_download
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [3]:
BASE = "/content/garbage_raw"
os.makedirs(BASE, exist_ok=True)

REPO_ID = "keremberke/garbage-object-detection"

for split in ["train", "valid", "test"]:
    zip_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=f"data/{split}.zip")
    out_dir = f"{BASE}/{split}"
    if not os.path.exists(out_dir):
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(out_dir)
    print(split, "->", out_dir, "(", len(os.listdir(out_dir)), "files )")

data/train.zip: reconstructing file:   0%|          |  0.00B /  138MB            

data/train.zip: downloading bytes:           |  0.00B            

train -> /content/garbage_raw/train ( 7325 files )


data/valid.zip: reconstructing file:   0%|          |  0.00B / 38.3MB            

data/valid.zip: downloading bytes:           |  0.00B            

valid -> /content/garbage_raw/valid ( 2099 files )


data/test.zip: reconstructing file:   0%|          |  0.00B / 19.6MB            

data/test.zip: downloading bytes:           |  0.00B            

test -> /content/garbage_raw/test ( 1043 files )


In [4]:
def coco_to_yolo(split_dir, out_images_dir, out_labels_dir, class_list=None):
    json_candidates = glob.glob(os.path.join(split_dir, '**', '*.json'), recursive=True)
    assert json_candidates, f"No COCO json found under {split_dir}"
    json_path = json_candidates[0]
    images_root = os.path.dirname(json_path)  # images usually live next to the json

    with open(json_path) as f:
        coco = json.load(f)

    categories = sorted(coco['categories'], key=lambda c: c['id'])
    if class_list is None:
        class_list = [c['name'] for c in categories]
    name_by_id = {c['id']: c['name'] for c in categories}
    cat_id_to_idx = {cid: class_list.index(name) for cid, name in name_by_id.items() if name in class_list}

    images_by_id = {img['id']: img for img in coco['images']}
    anns_by_image = {}
    for ann in coco['annotations']:
        anns_by_image.setdefault(ann['image_id'], []).append(ann)

    os.makedirs(out_images_dir, exist_ok=True)
    os.makedirs(out_labels_dir, exist_ok=True)

    n_written = 0
    for img_id, img in images_by_id.items():
        fname = img['file_name']
        src_path = os.path.join(images_root, fname)
        if not os.path.exists(src_path):
            continue
        shutil.copy(src_path, os.path.join(out_images_dir, fname))

        w, h = img['width'], img['height']
        lines = []
        for ann in anns_by_image.get(img_id, []):
            x, y, bw, bh = ann['bbox']
            cx, cy = (x + bw / 2) / w, (y + bh / 2) / h
            nw, nh = bw / w, bh / h
            cls_idx = cat_id_to_idx[ann['category_id']]
            lines.append(f"{cls_idx} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        label_name = os.path.splitext(fname)[0] + '.txt'
        with open(os.path.join(out_labels_dir, label_name), 'w') as f:
            f.write('\n'.join(lines))
        n_written += 1

    print(f"{split_dir}: wrote {n_written} images/labels -> classes={class_list}")
    return class_list


In [5]:
YOLO_DIR = "/content/garbage_yolo"

classes = coco_to_yolo(f"{BASE}/train", f"{YOLO_DIR}/train/images", f"{YOLO_DIR}/train/labels")
coco_to_yolo(f"{BASE}/valid", f"{YOLO_DIR}/valid/images", f"{YOLO_DIR}/valid/labels", class_list=classes)

data_yaml = f"""path: {YOLO_DIR}
train: train/images
val: valid/images
names: {classes}
"""
with open(f"{YOLO_DIR}/data.yaml", "w") as f:
    f.write(data_yaml)

print(data_yaml)

/content/garbage_raw/train: wrote 7324 images/labels -> classes=['biodegradable', 'cardboard', 'glass', 'metal', 'paper', 'plastic']
/content/garbage_raw/valid: wrote 2098 images/labels -> classes=['biodegradable', 'cardboard', 'glass', 'metal', 'paper', 'plastic']
path: /content/garbage_yolo
train: train/images
val: valid/images
names: ['biodegradable', 'cardboard', 'glass', 'metal', 'paper', 'plastic']



In [ ]:

RUNS_DIR = "/content/runs"

model = YOLO('yolov8n.pt')

model.train(
    data=f"{YOLO_DIR}/data.yaml",
    epochs=25,
    imgsz=416,
    batch=16,
    patience=8,       # early stop if val metric plateaus
    project=RUNS_DIR,
    name='garbage_yolov8n',
    exist_ok=True,
)

TRAIN_DIR = model.trainer.save_dir
print("Training outputs saved to:", TRAIN_DIR)


Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/garbage_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=garbage_yolov8n, nbs=64, nms=None, opset=None

In [ ]:
metrics = model.val(data=f"{YOLO_DIR}/data.yaml", project=RUNS_DIR, name='garbage_yolov8n_val', exist_ok=True)
VAL_DIR = metrics.save_dir

per_class_map50 = {
    classes[idx]: float(ap) for idx, ap in zip(metrics.box.ap_class_index, metrics.box.ap50)
}

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Per-class mAP50:", per_class_map50)
if len(per_class_map50) < len(classes):
    missing = set(classes) - set(per_class_map50)
    print("(no validation instances for:", missing, "- can't compute AP for these)")
print("Validation outputs saved to:", VAL_DIR)


In [ ]:
cm_path = os.path.join(str(VAL_DIR), 'confusion_matrix.png')
if os.path.exists(cm_path):
    plt.figure(figsize=(8,8))
    plt.imshow(mpimg.imread(cm_path))
    plt.axis('off')
    plt.show()
else:
    print("Not found at", cm_path, "- contents of VAL_DIR:", os.listdir(VAL_DIR))

In [ ]:
sample_images = glob.glob(f"{YOLO_DIR}/valid/images/*")[:6]
pred_results = model.predict(sample_images, conf=0.25, save=True, project=RUNS_DIR, name='garbage_preds', exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, r in zip(axes.flatten(), pred_results):
    ax.imshow(r.plot()[:, :, ::-1])  # BGR -> RGB
    ax.axis('off')
plt.tight_layout()
plt.show()

print("Prediction images saved to:", pred_results[0].save_dir)

In [ ]:
best_path = os.path.join(str(TRAIN_DIR), 'weights', 'best.pt')
shutil.copy(best_path, '/content/garbage_yolov8n_best.pt')
print("Saved to /content/garbage_yolov8n_best.pt")